# Federated comparison · multiclass · SGD

FedAvg, FedProx and SCAFFOLD use local SGD with initial learning rate 0.01 and zero momentum. The companion notebook without the `_sgd` suffix uses Adam and produced the archived tables.

In [ ]:
import json
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset


EXECUTION_DIR = Path.cwd().resolve()
ROOT_CANDIDATES = (
    EXECUTION_DIR,
    *EXECUTION_DIR.parents,
    EXECUTION_DIR / "uav-network-intrusion-detection",
)
ROOT = next(
    (
        candidato / "deep-learning-models"
        for candidato in ROOT_CANDIDATES
        if (candidato / "deep-learning-models" / "Normal+Attacks.csv").is_file()
    ),
    None,
)
if ROOT is None:
    ROOT = next(
        (candidato for candidato in ROOT_CANDIDATES if (candidato / "Normal+Attacks.csv").is_file()),
        None,
    )
if ROOT is None:
    raise FileNotFoundError("Could not locate deep-learning-models/Normal+Attacks.csv.")
REPO_ROOT = ROOT.parent

# ── Run configuration ──
# The seed identifies the random choices in this run. A comparison across methods also
# depends on using the same training and test examples.
SEED = 42

# ── Participation per round ──
# Five clients receive portions of the training set. One local epoch covers only that
# client’s portion.
NUM_CLIENTS = 5
ROUNDS = 10
LOCAL_EPOCHS = 5
BATCH_SIZE = 256

# ── Evaluation scale ──
# Here ε is expressed in standardized feature units. For a column with standard deviation s,
# 0.05 in this space corresponds to 0.05 × s in the original units.
FGSM_EPSILON = 0.05
ALGORITHMS = ["FedAvg", "FedProx", "SCAFFOLD"]
FEDPROX_MU = 0.01
LOCAL_LR = 0.01
ADV_WEIGHT = 0.5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE_MODEL_DIR = ROOT / "treated_multiclass_classification" / "models"
RESULTS_DIR = (
    REPO_ROOT
    / "adversarial-attacks"
    / "adversarial_results"
    / "fedavg_multiclass_sgd"
)


class MLPMulticlass(nn.Module):
    def __init__(self, input_dim, config, n_classes):
        super().__init__()
        hp = config["hiperparametros"]
        layers = []
        previous = input_dim
        activation = nn.ReLU if hp["ativacao"] == "relu" else nn.Tanh

        for index in range(hp["n_camadas"]):
            units = hp[f"unidades_{index}"]
            dropout = hp[f"dropout_{index}"]
            layers.extend((nn.Linear(previous, units), activation()))
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            previous = units

        layers.append(nn.Linear(previous, n_classes))
        self.rede = nn.Sequential(*layers)

    def forward(self, inputs):
        return self.rede(inputs)

Client indices refer to rows of the training set. Each partition aims to preserve class proportions. The aggregated model is evaluated on one common global test set. The five partitions simulate clients on one machine.

In [ ]:
def prepare_data(encoder):
    dataset_path = ROOT / "Normal+Attacks.csv"
    if not dataset_path.is_file():
        raise FileNotFoundError(f"Dataset not found: {dataset_path}")

    frame = pd.read_csv(dataset_path)
    if "label" not in frame.columns:
        raise ValueError(f"The dataset is missing the required column 'label': {dataset_path}")

    frame["label"] = frame["label"].replace(
        {"Reco": "Reco+Scanning", "Scanning": "Reco+Scanning"}
    )
    if (
        "fwd_seg_size_min" not in frame
        or frame["fwd_seg_size_min"].nunique(dropna=False) != 1
    ):
        raise ValueError(
            "Column fwd_seg_size_min is not part of the cleaned multiclass protocol."
        )
    frame = frame.drop(columns=["fwd_seg_size_min"])
    features = frame.drop(columns=["label"]).select_dtypes(include="number")
    if features.empty:
        raise ValueError("The dataset has no numeric features for training.")

    data = features.copy()
    data["target"] = frame["label"].astype(str).to_numpy()

    # ── Duplicate rows ──
    # The target is already in the table. Deduplication compares complete records. Grouping
    # later handles identical feature vectors retained because their labels differ.
    data = data.drop_duplicates().reset_index(drop=True)
    X = data[features.columns]
    class_names = encoder.classes_.tolist()

    # ── Class mapping ──
    # Each output index comes from the encoder used during training. An unknown label would
    # break the interpretation of checkpoint predictions.
    unknown_labels = set(data["target"]) - set(class_names)
    if unknown_labels:
        raise ValueError(
            f"Labels with no match in the saved encoder: {sorted(unknown_labels)}"
        )
    y = encoder.transform(data["target"]).astype("int64")

    # ── Groups for splitting ──
    # The hash depends on features and ignores the row index. Copies at different positions
    # receive the same group key.
    groups = pd.util.hash_pandas_object(X, index=False).to_numpy()
    holdout = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)
    train_indices, test_indices = next(holdout.split(X, y, groups=groups))

    scaler = StandardScaler()

    # ── Scale fitted on training data ──
    # Each column has its own mean and scale. The subsequent transform reuses these values
    # even when the test distribution differs.
    X_train = scaler.fit_transform(X.iloc[train_indices]).astype("float32")
    X_test = scaler.transform(X.iloc[test_indices]).astype("float32")
    y_train = y[train_indices]
    y_test = y[test_indices]
    return (
        X_train,
        y_train,
        X_test,
        y_test,
        scaler,
        features.columns.tolist(),
        class_names,
    )


def load_configuration():
    hp_path = ROOT / "bayesian_tuning" / "best_hyperparameters_summary_multiclass.json"
    metrics_path = ROOT / "treated_multiclass_classification" / "training_metrics.json"
    checkpoint_path = BASE_MODEL_DIR / "MLP_multiclass.pt"
    scaler_path = BASE_MODEL_DIR / "scaler_multiclass.joblib"
    encoder_path = BASE_MODEL_DIR / "label_encoder_multiclass.joblib"

    for path in (hp_path, metrics_path, checkpoint_path, scaler_path, encoder_path):
        if not path.is_file():
            raise FileNotFoundError(f"Required artifact not found: {path}")

    with hp_path.open(encoding="utf-8") as file:
        hyperparameters = json.load(file)["MLP"]
    with metrics_path.open(encoding="utf-8") as file:
        reference_metrics = json.load(file)["MLP"]

    return (
        hyperparameters,
        reference_metrics,
        checkpoint_path,
        joblib.load(scaler_path),
        joblib.load(encoder_path),
    )



# The saved configuration specifies the architecture. The checkpoint supplies weights
# learned by the reference model, which is evaluated on the same test rows.
def load_model(checkpoint_path, hyperparameters, input_dim, class_names):
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
    if int(checkpoint["n_features"]) != input_dim:
        raise ValueError("The number of checkpoint features does not match the data.")
    if int(checkpoint["n_classes"]) != len(class_names):
        raise ValueError("The number of checkpoint classes does not match the encoder.")
    if checkpoint["class_names"] != class_names:
        raise ValueError("The class order in the checkpoint differs from the encoder order.")

    model = MLPMulticlass(input_dim, hyperparameters, len(class_names)).to(DEVICE)
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    return model



# Indices refer to positions in X_train. The subsets represent simulated clients and
# approximate the training class distribution through stratification.
def create_client_partitions(X_train, y_train):
    splitter = StratifiedKFold(n_splits=NUM_CLIENTS, shuffle=True, random_state=SEED)
    partitions = [
        client_indices
        for _, client_indices in splitter.split(X_train, y_train)
    ]
    all_indices = np.concatenate(partitions)
    if len(all_indices) != len(X_train) or not np.array_equal(
        np.sort(all_indices), np.arange(len(X_train))
    ):
        raise RuntimeError("The client partition dropped or duplicated training examples.")
    return partitions


def generate_fgsm(model, inputs, targets):
    original = inputs.detach()
    adversarial_inputs = original.clone().requires_grad_(True)
    loss = nn.functional.cross_entropy(model(adversarial_inputs), targets)
    gradient = torch.autograd.grad(loss, adversarial_inputs)[0]
    return (original + FGSM_EPSILON * gradient.sign()).detach()

It receives global weights, processes its own batches and returns weights, sample count and timings. The server needs the count for weighted averaging. SCAFFOLD also returns an updated control that follows the client into the next round.

In [ ]:
def train_client(
    global_state, client_id, round_number, partition, X_train, y_train, hp, n_classes,
    algorithm="FedAvg", client_control=None, global_control=None
):

    # ── Start of a client round ──
    # The round and client together determine the seed. The same client therefore receives
    # the same random sequence in the corresponding round of each method.
    client_seed = SEED + round_number * 1000 + client_id
    torch.manual_seed(client_seed)
    np.random.seed(client_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(client_seed)

    local_model = MLPMulticlass(X_train.shape[1], hp, n_classes).to(DEVICE)

    # ── Starting weights ──
    # Weights come from the server. The optimizer is constructed immediately below, so its
    # internal state lasts only for this local training call.
    local_model.load_state_dict(global_state)

    # ── Local SGD without momentum ──
    # A corrected gradient of 0.3 at a learning rate of 0.01 reduces a weight by 0.003 in
    # that step. Every weight uses the same rate.
    optimizer = torch.optim.SGD(local_model.parameters(), lr=LOCAL_LR)
    criterion = nn.CrossEntropyLoss()
    generator = torch.Generator().manual_seed(client_seed)
    loader = DataLoader(
        TensorDataset(
            torch.as_tensor(X_train[partition], dtype=torch.float32),
            torch.as_tensor(y_train[partition], dtype=torch.int64),
        ),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
    )

    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)
    start_time = time.perf_counter()
    fgsm_seconds = 0.0
    optimizer_steps = 0
    for _ in range(LOCAL_EPOCHS):
        for clean_inputs, targets in loader:
            clean_inputs = clean_inputs.to(DEVICE)
            targets = targets.to(DEVICE)

            local_model.eval()
            if DEVICE.type == "cuda":
                torch.cuda.synchronize(DEVICE)
            fgsm_start = time.perf_counter()
            adversarial_inputs = generate_fgsm(local_model, clean_inputs, targets)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize(DEVICE)
            fgsm_seconds += time.perf_counter() - fgsm_start

            local_model.train()
            optimizer.zero_grad()
            clean_loss = criterion(local_model(clean_inputs), targets)
            adversarial_loss = criterion(local_model(adversarial_inputs), targets)

            # ── Batch objective ──
            # Both losses use the same targets. With ADV_WEIGHT = 0.5 each accounts for half
            # the loss before any FedProx penalty.
            loss = (1 - ADV_WEIGHT) * clean_loss + ADV_WEIGHT * adversarial_loss
            if algorithm == "FedProx":
                prox_loss = sum(
                    (parameter - global_state[name].to(DEVICE)).pow(2).sum()
                    for name, parameter in local_model.named_parameters()
                )
                loss = loss + 0.5 * FEDPROX_MU * prox_loss
            loss.backward()
            if algorithm == "SCAFFOLD" and client_control is not None and global_control is not None:
                for name, parameter in local_model.named_parameters():
                    if parameter.grad is not None:

                        # ── SCAFFOLD ──
                        # Backpropagation has already computed the batch gradient. The
                        # correction c − cᵢ enters before the optimizer step, while its
                        # direction can still be changed.
                        parameter.grad.add_(
                            global_control[name].to(DEVICE) - client_control[name].to(DEVICE)
                        )
            optimizer.step()
            optimizer_steps += 1

    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)
    elapsed_seconds = time.perf_counter() - start_time
    local_state = {
        name: tensor.detach().cpu().clone()
        for name, tensor in local_model.state_dict().items()
    }
    updated_control = None
    if algorithm == "SCAFFOLD" and client_control is not None and global_control is not None:
        learning_rate = LOCAL_LR

        # ── Control for the next round ──
        # K counts batch updates. With fixed rate η, (initial − final weight)/(Kη) recovers
        # the average corrected gradient used by SGD. The cᵢ − c term completes the local
        # control update.
        denominator = max(1, optimizer_steps) * learning_rate
        updated_control = {}
        for name, local_tensor in local_state.items():
            updated_control[name] = (
                client_control[name].to(DEVICE)
                - global_control[name].to(DEVICE)
                + (global_state[name].to(DEVICE) - local_tensor.to(DEVICE)) / denominator
            ).detach().cpu()
    return local_state, len(partition), elapsed_seconds, fgsm_seconds, updated_control


def agregar_fedavg(client_states, sample_counts):
    if not client_states or len(client_states) != len(sample_counts):
        raise ValueError("Client states and sample counts must have the same length.")

    # ── Client weights ──
    # A client holding 200 of 1,000 examples contributes 20% of the average. With full
    # participation the weights sum to one.
    total_samples = sum(sample_counts)
    if total_samples <= 0 or any(count <= 0 for count in sample_counts):
        raise ValueError("Every participating client must have samples to aggregate.")

    aggregated_state = {}
    for name in client_states[0]:
        aggregated_state[name] = torch.zeros_like(client_states[0][name])
        for state, count in zip(client_states, sample_counts):
            aggregated_state[name].add_(state[name], alpha=count / total_samples)
    return aggregated_state

def aggregate_scaffold_controls(global_control, client_controls, sample_counts):

    # ── Client weights ──
    # A client holding 200 of 1,000 examples contributes 20% of the average. With full
    # participation the weights sum to one.
    total_samples = sum(sample_counts)
    updated_control = {}
    for name in global_control:
        delta = torch.zeros_like(global_control[name])
        for control, count in zip(client_controls, sample_counts):
            delta.add_(control[name] - global_control[name], alpha=count / total_samples)
        updated_control[name] = global_control[name] + delta
    return updated_control

Accuracy counts all correct predictions. Macro-F1 gives equal weight to each class F1. For ASR, align clean and perturbed predictions from the same model by row. If it originally got 80 examples right and now misses 8 of those, ASR is 8/80 = 10%.

In [ ]:
def predict(model, inputs):
    model.eval()
    predictions = []
    with torch.no_grad():
        for start in range(0, len(inputs), 1024):
            batch = torch.as_tensor(
                inputs[start : start + 1024], dtype=torch.float32, device=DEVICE
            )
            predictions.extend(model(batch).argmax(dim=1).cpu().tolist())
    return np.asarray(predictions, dtype="int64")


def generate_test_fgsm(model, X_values, y_values):
    adversarial_batches = []
    model.eval()
    for start in range(0, len(X_values), BATCH_SIZE):
        inputs = torch.as_tensor(
            X_values[start : start + BATCH_SIZE], dtype=torch.float32, device=DEVICE
        )
        targets = torch.as_tensor(
            y_values[start : start + BATCH_SIZE], dtype=torch.int64, device=DEVICE
        )
        adversarial_batches.append(generate_fgsm(model, inputs, targets).cpu().numpy())
    return np.concatenate(adversarial_batches, axis=0)


def evaluate_metrics(y_true, y_pred, class_names, y_clean_pred=None):
    labels = np.arange(len(class_names))
    matrix = confusion_matrix(y_true, y_pred, labels=labels)
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(
            f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
        ),
        "confusion_matrix": matrix.tolist(),
        "classification_report": classification_report(
            y_true,
            y_pred,
            labels=labels,
            target_names=class_names,
            output_dict=True,
            zero_division=0,
        ),
    }
    if y_clean_pred is None:
        metrics["attack_success_rate"] = None
    else:

        # ── ASR denominator ──
        # Only examples this model got right on clean inputs are eligible. Ten new errors
        # among 80 initially correct predictions give 12.5% ASR.
        initially_correct = y_clean_pred == y_true
        denominator = int(initially_correct.sum())
        metrics["attack_success_rate"] = (
            float(np.logical_and(initially_correct, y_pred != y_true).sum() / denominator)
            if denominator
            else None
        )
        metrics["attack_success_definition"] = (
            "Fraction of clean-correct test samples misclassified after untargeted FGSM."
        )
    return metrics



# The denominator is the reference model’s drop under perturbation. A small drop can
# turn a modest difference into a large recovery percentage.
def recovery(clean_score, attacked_score, defended_score):
    drop = clean_score - attacked_score
    return (defended_score - attacked_score) / drop if drop > 0 else None

A round begins with a common model snapshot. The server waits for every client before aggregating. With ten rounds and five local epochs, each client traverses its partition fifty times. Sequential client execution affects the measured runtime.

In [ ]:
def main():
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    (
        hyperparameters,
        reference_metrics,
        checkpoint_path,
        saved_scaler,
        encoder,
    ) = load_configuration()
    (
        X_train,
        y_train,
        X_test,
        y_test,
        scaler,
        feature_names,
        class_names,
    ) = prepare_data(encoder)
    if not np.allclose(saved_scaler.mean_, scaler.mean_) or not np.allclose(
        saved_scaler.scale_, scaler.scale_
    ):
        raise ValueError("The reconstructed scaler does not match the saved scaler.")

    noadv_model = load_model(
        checkpoint_path, hyperparameters, X_train.shape[1], class_names
    )
    noadv_clean = predict(noadv_model, X_test)
    noadv_clean_metrics = evaluate_metrics(y_test, noadv_clean, class_names)
    for metric_name in ("accuracy", "macro_f1"):
        if not np.isclose(
            noadv_clean_metrics[metric_name],
            float(reference_metrics[metric_name]),
            atol=1e-12,
        ):
            raise ValueError(
                f"The NoAdv {metric_name} metric differs from the reference artifact; "
                "check the split, scaler, encoder, and checkpoint."
            )

    with torch.enable_grad():
        noadv_fgsm = generate_test_fgsm(noadv_model, X_test, y_test)
    noadv_attacked = predict(noadv_model, noadv_fgsm)

    partitions = create_client_partitions(X_train, y_train)
    for client_id, indices in enumerate(partitions, start=1):
        counts = np.bincount(y_train[indices], minlength=len(class_names))
        print(f"Simulated client {client_id}: {len(indices):,} samples; classes={counts.tolist()}")

    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    global_model = MLPMulticlass(
        X_train.shape[1], hyperparameters, len(class_names)
    ).to(DEVICE)
    global_state = {
        name: tensor.detach().cpu().clone()
        for name, tensor in global_model.state_dict().items()
    }
    model_payload_bytes = sum(
        tensor.numel() * tensor.element_size() for tensor in global_state.values()
    )

    # Round history. The newly aggregated state becomes the starting point for every client
    # in the next round.
    round_history = []
    local_training_seconds = 0.0
    fgsm_generation_seconds = 0.0
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)
    federated_start = time.perf_counter()
    for round_number in range(1, ROUNDS + 1):
        client_states = []
        sample_counts = []
        round_fgsm_seconds = 0.0
        for client_id, partition in enumerate(partitions):
            local_state, count, elapsed, fgsm_seconds, _ = train_client(
                global_state,
                client_id,
                round_number,
                partition,
                X_train,
                y_train,
                hyperparameters,
                len(class_names),
            )
            client_states.append(local_state)
            sample_counts.append(count)
            local_training_seconds += elapsed
            round_fgsm_seconds += fgsm_seconds

        global_state = agregar_fedavg(client_states, sample_counts)
        global_model.load_state_dict(global_state)
        fgsm_generation_seconds += round_fgsm_seconds
        round_history.append(
            {
                "round": round_number,
                "participating_clients": len(client_states),
                "examples_aggregated": int(sum(sample_counts)),
                "fgsm_generation_seconds": round_fgsm_seconds,
            }
        )
        print(
            f"Round {round_number}/{ROUNDS}: "
            f"{len(client_states)} clients, {sum(sample_counts):,} aggregated examples."
        )

    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)
    federated_wall_seconds = time.perf_counter() - federated_start
    fedavg_clean = predict(global_model, X_test)
    with torch.enable_grad():
        fedavg_fgsm = generate_test_fgsm(global_model, X_test, y_test)
    fedavg_attacked = predict(global_model, fedavg_fgsm)

    # ── Evaluation conditions ──
    # Clean and perturbed predictions are paired by example. Their difference measures the
    # sensitivity of the evaluated model.
    conditions = {
        "NoAdv_clean_test": evaluate_metrics(y_test, noadv_clean, class_names),
        "NoAdv_FGSM_test": evaluate_metrics(
            y_test, noadv_attacked, class_names, noadv_clean
        ),
        "FedAvg_FGSM_clean_test": evaluate_metrics(y_test, fedavg_clean, class_names),
        "FedAvg_FGSM_FGSM_test": evaluate_metrics(
            y_test, fedavg_attacked, class_names, fedavg_clean
        ),
    }
    accuracy_recovery = recovery(
        conditions["NoAdv_clean_test"]["accuracy"],
        conditions["NoAdv_FGSM_test"]["accuracy"],
        conditions["FedAvg_FGSM_FGSM_test"]["accuracy"],
    )
    macro_f1_recovery = recovery(
        conditions["NoAdv_clean_test"]["macro_f1"],
        conditions["NoAdv_FGSM_test"]["macro_f1"],
        conditions["FedAvg_FGSM_FGSM_test"]["macro_f1"],
    )

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    model_dir = RESULTS_DIR / "models"
    model_dir.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "state_dict": global_model.state_dict(),
            "local_optimizer": "SGD",            "local_learning_rate": LOCAL_LR,            "algorithm": "FedAvg",
            "training_attack": "untargeted FGSM",
            "n_classes": len(class_names),
            "n_features": len(feature_names),
            "class_names": class_names,
            "feature_names": feature_names,
        },
        model_dir / "MLP_FedAvg_FGSM_multiclass.pt",
    )
    joblib.dump(scaler, model_dir / "scaler_multiclass.joblib")
    joblib.dump(encoder, model_dir / "label_encoder_multiclass.joblib")

    # ── Communication cost ──
    # The estimate counts tensor bytes sent and received. Measured traffic would also include
    # protocol overhead and any retransmissions.
    bytes_per_client_round = 2 * model_payload_bytes
    results = {
        "configuration": {
            "dataset": "Normal+Attacks.csv",
            "scenario": "multiclass_cleaned",
            "local_optimizer": "SGD",            "local_learning_rate": LOCAL_LR,            "algorithm": "FedAvg",
            "local_adversarial_training": True,
            "client_simulation": (
                "5 stratified IID shards from training data; no source/client IDs in dataset"
            ),
            "split": "first StratifiedGroupKFold split of 4; groups by feature vector",
            "all_clients_participate_each_round": True,
            "seed": SEED,
            "clients": NUM_CLIENTS,
            "rounds": ROUNDS,
            "local_epochs": LOCAL_EPOCHS,
            "batch_size": BATCH_SIZE,
            "epsilon": FGSM_EPSILON,
            "epsilon_unit": "standardized feature space (StandardScaler)",
            "adversarial_weight": ADV_WEIGHT,
            "classes": class_names,
            "training_samples": int(len(y_train)),
            "test_samples": int(len(y_test)),
            "test_class_counts": {
                name: int((y_test == index).sum())
                for index, name in enumerate(class_names)
            },
            "device": str(DEVICE),
            "limitacao": (
                "Single-seed centralized simulation; clients are not real devices and "
                "features perturbations are not packet-level attacks."
            ),
        },
        "metrics": conditions,
        "relative_accuracy_recovery": accuracy_recovery,
        "relative_macro_f1_recovery": macro_f1_recovery,
        "cost": {
            "federated_training_wall_time_seconds": federated_wall_seconds,
            "sum_local_training_time_seconds": local_training_seconds,
            "fgsm_generation_time_seconds": fgsm_generation_seconds,
            "model_payload_bytes": model_payload_bytes,
            "estimated_bytes_per_client_per_round": bytes_per_client_round,
            "estimated_total_bytes": (
                bytes_per_client_round * NUM_CLIENTS * ROUNDS
            ),
            "note": (
                "Estimated model weights sent downlink and uplink; excludes protocol, "
                "metadata and latency. Clients execute sequentially in one process."
            ),
        },
        "round_history": round_history,
    }
    output_path = RESULTS_DIR / "fedavg_multiclass_metrics.json"
    output_path.write_text(
        json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8"
    )

    print(f"Results saved to: {output_path}")
    for condition, metrics in conditions.items():
        print(
            f"{condition}: accuracy={metrics['accuracy']:.4f}, "
            f"macro_f1={metrics['macro_f1']:.4f}, "
            f"attack_success_rate={metrics['attack_success_rate']}"
        )
    print(f"Relative accuracy recovery: {accuracy_recovery}")
    print(f"Relative macro-F1 recovery: {macro_f1_recovery}")

    # ── Results by method ──
    # FedAvg has already been evaluated. Subsequent entries use the same test rows and add
    # each model’s metrics to the same report.
    comparison_results = {"FedAvg": results}
    for algorithm in ("FedProx", "SCAFFOLD"):
        torch.manual_seed(SEED)
        np.random.seed(SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(SEED)
        global_model = MLPMulticlass(X_train.shape[1], hyperparameters, len(class_names)).to(DEVICE)
        algorithm_global_state = {
            name: tensor.detach().cpu().clone()
            for name, tensor in global_model.state_dict().items()
        }
        client_controls = None
        global_control = None

        # Controls start at zero. Each client carries its own control into the next round,
        # while the server maintains the global control.
        if algorithm == "SCAFFOLD":
            client_controls = [
                {name: torch.zeros_like(tensor) for name, tensor in algorithm_global_state.items()}
                for _ in partitions
            ]
            global_control = {
                name: torch.zeros_like(tensor)
                for name, tensor in algorithm_global_state.items()
            }
        algorithm_history = []
        algorithm_training_seconds = 0.0
        algorithm_fgsm_seconds = 0.0
        algorithm_start = time.perf_counter()
        for round_number in range(1, ROUNDS + 1):
            client_states = []
            sample_counts = []
            updated_controls = []
            round_fgsm_seconds = 0.0
            for client_id, partition in enumerate(partitions):
                local_control = client_controls[client_id] if client_controls is not None else None
                local_state, count, elapsed, client_fgsm, updated_control = train_client(
                    algorithm_global_state,
                    client_id,
                    round_number,
                    partition,
                    X_train,
                    y_train,
                    hyperparameters,
                    len(class_names),
                    algorithm=algorithm,
                    client_control=local_control,
                    global_control=global_control,
                )
                client_states.append(local_state)
                sample_counts.append(count)
                if updated_control is not None:
                    updated_controls.append(updated_control)
                algorithm_training_seconds += elapsed
                algorithm_fgsm_seconds += client_fgsm
                round_fgsm_seconds += client_fgsm
            algorithm_global_state = agregar_fedavg(client_states, sample_counts)
            if algorithm == "SCAFFOLD" and updated_controls:
                client_controls = updated_controls
                global_control = aggregate_scaffold_controls(global_control, updated_controls, sample_counts)
            global_model.load_state_dict(algorithm_global_state)
            algorithm_history.append({
                "round": round_number,
                "participating_clients": len(client_states),
                "examples_aggregated": int(sum(sample_counts)),
                "fgsm_generation_seconds": round_fgsm_seconds,
            })
            print(f"{algorithm} Round {round_number}/{ROUNDS}: {sum(sample_counts):,} examples.")
        algorithm_wall_seconds = time.perf_counter() - algorithm_start
        clean_predictions = predict(global_model, X_test)
        with torch.enable_grad():
            fgsm_predictions = generate_test_fgsm(global_model, X_test, y_test)
        attacked_predictions = predict(global_model, fgsm_predictions)
        clean_metrics = evaluate_metrics(y_test, clean_predictions, class_names)
        attacked_metrics = evaluate_metrics(y_test, attacked_predictions, class_names, clean_predictions)
        accuracy_recovery = recovery(
            conditions["NoAdv_clean_test"]["accuracy"],
            conditions["NoAdv_FGSM_test"]["accuracy"],
            attacked_metrics["accuracy"],
        )
        macro_f1_recovery = recovery(
            conditions["NoAdv_clean_test"]["macro_f1"],
            conditions["NoAdv_FGSM_test"]["macro_f1"],
            attacked_metrics["macro_f1"],
        )
        control_payload_bytes = sum(
            tensor.numel() * tensor.element_size() for tensor in global_control.values()
        ) if algorithm == "SCAFFOLD" else 0

        # ── Communication cost ──
        # The estimate counts tensor bytes sent and received. Measured traffic would also
        # include protocol overhead and any retransmissions.
        communication_bytes = 2 * (model_payload_bytes + control_payload_bytes) * NUM_CLIENTS * ROUNDS
        algorithm_results = {
            "configuration": {
                "dataset": "Normal+Attacks.csv",
                "scenario": "multiclass_cleaned",
                "local_optimizer": "SGD",                "local_learning_rate": LOCAL_LR,                "algorithm": algorithm,
                "fedprox_mu": FEDPROX_MU if algorithm == "FedProx" else None,
                "local_adversarial_training": True,
                "seed": SEED,
                "clients": NUM_CLIENTS,
                "rounds": ROUNDS,
                "local_epochs": LOCAL_EPOCHS,
                "batch_size": BATCH_SIZE,
                "epsilon": FGSM_EPSILON,
                "adversarial_weight": ADV_WEIGHT,
                "classes": class_names,
            },
            "metrics": {
                "NoAdv_clean_test": conditions["NoAdv_clean_test"],
                f"{algorithm}_FGSM_clean_test": clean_metrics,
                f"{algorithm}_FGSM_FGSM_test": attacked_metrics,
            },
            "relative_accuracy_recovery": accuracy_recovery,
            "relative_macro_f1_recovery": macro_f1_recovery,
            "cost": {
                "federated_training_wall_time_seconds": algorithm_wall_seconds,
                "sum_local_training_time_seconds": algorithm_training_seconds,
                "fgsm_generation_time_seconds": algorithm_fgsm_seconds,
                "model_payload_bytes": model_payload_bytes,
                "scaffold_control_payload_bytes": control_payload_bytes,
                "estimated_total_bytes": communication_bytes,
            },
            "round_history": algorithm_history,
        }
        comparison_results[algorithm] = algorithm_results
        torch.save({
            "state_dict": global_model.state_dict(),
            "local_optimizer": "SGD",            "local_learning_rate": LOCAL_LR,            "algorithm": algorithm,
            "training_attack": "untargeted FGSM",
            "n_classes": len(class_names),
            "n_features": len(feature_names),
            "class_names": class_names,
            "feature_names": feature_names,
        }, model_dir / f"MLP_{algorithm}_FGSM_multiclass.pt")
        (RESULTS_DIR / f"{algorithm.lower()}_multiclass_metrics.json").write_text(
            json.dumps(algorithm_results, indent=2, ensure_ascii=False), encoding="utf-8"
        )
    comparison_path = RESULTS_DIR / "federated_comparison_multiclass.json"
    comparison_path.write_text(json.dumps(comparison_results, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Federated comparison saved to: {comparison_path}")

The routine prepares the data, evaluates the reference, trains and saves results. A full run can overwrite files for the same experiment.

In [ ]:
if __name__ == "__main__":
    main()

## Report from saved JSON

This cell reads the saved results independently.

This cell reads the training JSON and rebuilds the tables and figures. Running this cell alone updates the presentation of the same results.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = next(
    (candidate for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (candidate / "deep-learning-models" / "Normal+Attacks.csv").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not locate the repository root.")
RESULTS_DIR = REPO_ROOT / "adversarial-attacks" / "adversarial_results" / "fedavg_multiclass_sgd"

comparison_path = RESULTS_DIR / "federated_comparison_multiclass.json"
comparison = json.loads(comparison_path.read_text(encoding="utf-8"))
report_dir = REPO_ROOT / "graphics" / "federated_comparison_sgd" / "multiclass"
report_dir.mkdir(parents=True, exist_ok=True)

methods = ["FedAvg", "FedProx", "SCAFFOLD"]
noadv = comparison["FedAvg"]["metrics"]
noadv_clean = noadv["NoAdv_clean_test"]
noadv_attacked = noadv["NoAdv_FGSM_test"]

# ── Centralized reference ──
# The first record comes from NoAdv. It appears once in the table even when its result is
# stored alongside several methods.
summary_rows = [{
    "Method": "NoAdv",
    "Clean accuracy": noadv_clean["accuracy"],
    "FGSM accuracy": noadv_attacked["accuracy"],
    "Clean macro-F1": noadv_clean["macro_f1"],
    "FGSM macro-F1": noadv_attacked["macro_f1"],
    "FGSM ASR": noadv_attacked["attack_success_rate"],
}]
recovery_rows = []

# ── Time and volume ──
# Timings come from this run. Byte counts are calculated from tensor sizes and number of
# rounds.
cost_rows = []
per_class_rows = []

for method in methods:
    result = comparison[method]
    clean_key = f"{method}_FGSM_clean_test"
    attacked_key = f"{method}_FGSM_FGSM_test"
    clean = result["metrics"][clean_key]
    attacked = result["metrics"][attacked_key]
    summary_rows.append({
        "Method": method,
        "Clean accuracy": clean["accuracy"],
        "FGSM accuracy": attacked["accuracy"],
        "Clean macro-F1": clean["macro_f1"],
        "FGSM macro-F1": attacked["macro_f1"],
        "FGSM ASR": attacked["attack_success_rate"],
    })
    recovery_rows.append({
        "Method": method,
        "Relative accuracy recovery": result["relative_accuracy_recovery"],
        "Relative macro-F1 recovery": result["relative_macro_f1_recovery"],
    })
    cost = result["cost"]
    cost_rows.append({
        "Method": method,
        "Training wall time (s)": cost["federated_training_wall_time_seconds"],
        "FGSM generation time (s)": cost["fgsm_generation_time_seconds"],
        "Model payload (bytes)": cost["model_payload_bytes"],
        "Estimated communication (bytes)": cost["estimated_total_bytes"],
    })
    report = attacked["classification_report"]
    for class_name, values in report.items():
        if class_name in ("accuracy", "macro avg", "weighted avg"):
            continue
        per_class_rows.append({
            "Method": method,
            "Class": class_name,
            "Precision": values["precision"],
            "Recall": values["recall"],
            "F1": values["f1-score"],
            "Support": int(values["support"]),
        })

summary = pd.DataFrame(summary_rows)
recovery = pd.DataFrame(recovery_rows)
costs = pd.DataFrame(cost_rows)
per_class = pd.DataFrame(per_class_rows)
summary.to_csv(report_dir / "multiclass_federated_summary.csv", index=False)
recovery.to_csv(report_dir / "multiclass_relative_recovery.csv", index=False)
costs.to_csv(report_dir / "multiclass_federated_cost.csv", index=False)
per_class.to_csv(report_dir / "multiclass_per_class_fgsm.csv", index=False)

display(summary.round(6))
display(recovery.round(6))
display(per_class.round(6))
display(costs)

plt.style.use("default")
colors = {"NoAdv": "#6B7280", "FedAvg": "#2563EB", "FedProx": "#DC2626", "SCAFFOLD": "#16A34A"}

summary_indexed = summary.set_index("Method")

for metric_columns, title, ylabel, filename in [
    (["Clean accuracy", "FGSM accuracy"], "Multiclass federated methods: accuracy", "Accuracy", "multiclass_accuracy_comparison.png"),
    (["Clean macro-F1", "FGSM macro-F1"], "Multiclass federated methods: macro-F1", "Macro-F1", "multiclass_macro_f1_comparison.png"),
]:
    ax = summary_indexed[list(metric_columns)].plot(kind="bar", figsize=(9, 5), rot=0)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", alpha=0.25)
    ax.legend(metric_columns, frameon=False)
    plt.tight_layout()
    plt.savefig(report_dir / filename, dpi=150, bbox_inches="tight")
    plt.show()

ax = summary_indexed["FGSM ASR"].plot(
    kind="bar", figsize=(7, 5), color=[colors[m] for m in summary["Method"]], rot=0
)
ax.set_title("Multiclass federated methods: FGSM attack success rate")
ax.set_ylabel("Attack success rate")
ax.set_xlabel("")
ax.set_ylim(0, max(0.01, float(summary["FGSM ASR"].max()) * 1.25))
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(report_dir / "multiclass_asr_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

ax = recovery.set_index("Method")[["Relative accuracy recovery", "Relative macro-F1 recovery"]].plot(
    kind="bar", figsize=(9, 5), rot=0
)
ax.axhline(0, color="#111827", linewidth=1)
ax.set_title("Relative recovery under FGSM")
ax.set_ylabel("Relative recovery")
ax.set_xlabel("")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(report_dir / "multiclass_relative_recovery.png", dpi=150, bbox_inches="tight")
plt.show()

pivot_f1 = per_class.pivot(index="Class", columns="Method", values="F1")
ax = pivot_f1.plot(kind="bar", figsize=(12, 6), rot=45)
ax.set_title("Per-class F1 under FGSM")
ax.set_ylabel("F1-score")
ax.set_xlabel("")
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(report_dir / "multiclass_per_class_f1_under_fgsm.png", dpi=150, bbox_inches="tight")
plt.show()

class_names = comparison["FedAvg"]["configuration"]["classes"]
for method in methods:
    result = comparison[method]
    attacked_key = f"{method}_FGSM_FGSM_test"
    matrix = np.asarray(result["metrics"][attacked_key]["confusion_matrix"])
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(matrix, cmap="Blues")
    ax.set_xticks(np.arange(len(class_names)), labels=class_names, rotation=45, ha="right")
    ax.set_yticks(np.arange(len(class_names)), labels=class_names)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{method}: confusion matrix under FGSM")
    row_totals = matrix.sum(axis=1, keepdims=True)
    percentages = np.divide(matrix, row_totals, out=np.zeros_like(matrix, dtype=float), where=row_totals != 0)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, f"{matrix[i, j]}\n{percentages[i, j]:.1%}", ha="center", va="center", fontsize=7)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.savefig(report_dir / f"multiclass_confusion_matrix_{method.lower()}_fgsm.png", dpi=150, bbox_inches="tight")
    plt.show()